# Dukascopy scratch: extension Parquet + 2021 gap fill

**Ephemeral notebook** — copy of the ingest helpers; safe to delete when you are done.

**Recommended order (matches your plan)**
1. Backfill **2025-11-05 → 2026-03-27** into **daily** Parquets under `dukascopy/` (and patch manifest/report):  
   `python scripts/fix_dukascopy_gap_dates.py --extension-only`  
   Or run section **1** below (writes one extension Parquet under `scratch_notebook_outputs/` instead — pick one approach).
2. **Split** your huge combined file to dailies if needed (`scripts/split_combined_ticks_to_dailies.py`), then backfill **2021-06-11**:  
   `python scripts/fix_dukascopy_gap_dates.py --extra-only`
3. **Combine** later (split → merge all dailies in the ingest notebook, or DuckDB — decide after both steps succeed).

**Manifest / `validation_report.csv`:** this notebook does **not** open or change them. It only writes Parquets (and optional small CSVs) under `scratch_notebook_outputs/`. Your main manifest and report stay exactly as they are.

1. **Extension** — download a calendar range (default late 2025 through early 2026), validate like the main pipeline, write **one new Parquet** (`EURUSD_ticks_extension_*.parquet`). Chronologically this is an *append* to your history, not an insert.

2. **Gap fill (why two neighbor files?)** — Time order is … June 9, **June 10**, **June 11**, **June 12**, June 13 … If June 11 was missing, we rebuild a **short continuous window**: take ticks you already have for June 10, add freshly downloaded June 11, add ticks you already have for June 12, then sort. That needs **two** inputs on disk (June 10 and June 12 daily Parquets). The result is **one new file** covering only those three days — it does not rewrite your big combined file.

Merging these into your main combined file is a separate step (full re-merge, DuckDB union + sort, or keep as extra files and union at query time).

In [ ]:
from __future__ import annotations

import lzma
import struct
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
# --- Tunables -----------------------------------------------------------
SYMBOL = "EURUSD"
PRICE_SCALE = 1e5

HTTP_TIMEOUT_SEC = 60
HTTP_RETRIES = 5
HTTP_RETRY_BACKOFF_SEC = 0.8
DELAY_BETWEEN_HOURS_SEC = 0.05

# --- Validation thresholds (required by validate_day below; same meaning as ingest_dukascopy.ipynb)
# Plausible EUR/USD quote range for bid/ask (reject obvious garbage)
PRICE_MIN = 0.5
PRICE_MAX = 2.5
# ask - bid: warn above SPREAD_MAX_OK; error above SPREAD_MAX_ERR (rates in price units, not pips)
SPREAD_MAX_OK = 0.005
SPREAD_MAX_ERR = 0.020

# Forward extension (late 2025 / early 2026 — edit as needed)
EXTENSION_START = "2025-11-05"
EXTENSION_END = "2026-03-27"

# 2021 gap: insert this day between its two UTC neighbors (requires neighbor Parquets on disk)
GAP_DAY = "2021-06-11"
NEIGHBOR_BEFORE = "2021-06-10"
NEIGHBOR_AFTER = "2021-06-12"


def _repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Cannot locate project root.")


ROOT = _repo_root()
OUT_DIR = ROOT / "data" / "dirty" / "ticks" / "raw_source" / "dukascopy"
CAL_PATH = ROOT / "data" / "dirty" / "calendar" / "us_calendar.parquet"
SCRATCH_DIR = OUT_DIR / "scratch_notebook_outputs"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

tag_ext = EXTENSION_START.replace("-", "") + "_" + EXTENSION_END.replace("-", "")
EXT_PARQUET = SCRATCH_DIR / f"EURUSD_ticks_extension_{tag_ext}.parquet"
GAP_PARQUET = SCRATCH_DIR / f"EURUSD_ticks_{NEIGHBOR_BEFORE}_to_{NEIGHBOR_AFTER}_gapfill.parquet"

print("OUT_DIR   :", OUT_DIR)
print("SCRATCH   :", SCRATCH_DIR)
print("Extension :", EXT_PARQUET.name)
print("Gap fill  :", GAP_PARQUET.name)

In [ ]:
cal = pd.read_parquet(CAL_PATH)
cal["date"] = pd.to_datetime(cal["date"]).dt.normalize()
cal = cal.set_index("date")
print(f"Calendar loaded: {len(cal)} days")

In [ ]:
def dukascopy_url(symbol: str, day: datetime, hour: int) -> str:
    return (
        f"https://datafeed.dukascopy.com/datafeed/{symbol}/"
        f"{day.year}/{day.month - 1:02d}/{day.day:02d}/{hour:02d}h_ticks.bi5"
    )


def decode_bi5(blob: bytes, hour_start: datetime) -> pd.DataFrame:
    raw = lzma.decompress(blob)
    rec_size = 20
    n = len(raw) // rec_size
    rows = []
    for i in range(n):
        ms, ask_i, bid_i, ask_vol, bid_vol = struct.unpack_from(">IIIff", raw, i * rec_size)
        ts = hour_start + timedelta(milliseconds=int(ms))
        rows.append((ts, bid_i / PRICE_SCALE, ask_i / PRICE_SCALE, ask_vol, bid_vol))
    return pd.DataFrame(rows, columns=["timestamp_utc", "bid", "ask", "bid_volume", "ask_volume"])


def fetch_bi5_hour(url: str) -> bytes | None:
    req = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            )
        },
    )
    backoff = HTTP_RETRY_BACKOFF_SEC
    for attempt in range(HTTP_RETRIES):
        try:
            with urlopen(req, timeout=HTTP_TIMEOUT_SEC) as resp:
                blob = resp.read()
            if blob:
                return blob
        except Exception:
            pass
        if attempt < HTTP_RETRIES - 1:
            time.sleep(backoff * (2**attempt))
    return None


def download_day(symbol: str, day_str: str) -> tuple[pd.DataFrame, list[int]]:
    day = datetime.strptime(day_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    chunks: list[pd.DataFrame] = []
    missing_hours: list[int] = []

    for hour in range(24):
        hour_start = day + timedelta(hours=hour)
        url = dukascopy_url(symbol, day, hour)
        blob = fetch_bi5_hour(url)
        if not blob:
            missing_hours.append(hour)
            if DELAY_BETWEEN_HOURS_SEC:
                time.sleep(DELAY_BETWEEN_HOURS_SEC)
            continue
        try:
            chunks.append(decode_bi5(blob, hour_start))
        except Exception as exc:
            missing_hours.append(hour)
            print(f"  [WARN] decode {day_str} h{hour:02d}: {exc}")
        if DELAY_BETWEEN_HOURS_SEC:
            time.sleep(DELAY_BETWEEN_HOURS_SEC)

    if not chunks:
        df = pd.DataFrame(columns=["timestamp_utc", "bid", "ask", "bid_volume", "ask_volume"])
    else:
        df = pd.concat(chunks, ignore_index=True)
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
    return df, missing_hours

In [ ]:
def validate_day(
    df: pd.DataFrame,
    day_str: str,
    missing_hours: list[int],
    cal: pd.DataFrame,
) -> tuple[str, list[str], list[dict]]:
    issues: list[dict] = []
    warn_codes: list[str] = []

    day_ts = pd.Timestamp(day_str)
    cal_row = cal.loc[day_ts] if day_ts in cal.index else None

    is_weekend = bool(cal_row["is_weekend"]) if cal_row is not None else None
    is_us_holiday = bool(cal_row["is_us_holiday"]) if cal_row is not None else None
    is_expected = bool(cal_row["is_expected_data"]) if cal_row is not None else None
    day_of_week = int(cal_row["day_of_week"]) if cal_row is not None else None
    is_friday = day_of_week == 4

    friday_eod_hours = {21, 22, 23}
    missing_set = set(missing_hours)
    missing_hours_expected = (
        bool(is_weekend)
        or bool(is_us_holiday)
        or (is_friday and missing_set.issubset(friday_eod_hours))
    )

    def add(issue: str, detail: str, severity: str) -> None:
        warn_codes.append(issue)
        issues.append({"date": day_str, "issue": issue, "detail": detail, "severity": severity})

    rows = len(df)

    if rows == 0 and is_expected and not is_weekend and not is_us_holiday:
        add(
            "empty_expected_day",
            f"0 rows on expected-data day (weekend={is_weekend}, holiday={is_us_holiday})",
            "error",
        )

    if len(missing_hours) > 0 and not missing_hours_expected:
        sev = "warn" if len(missing_hours) <= 6 else "error"
        add("missing_hours", f"{len(missing_hours)} hours missing: {missing_hours}", sev)

    if rows == 0:
        if any(i["severity"] == "error" for i in issues):
            status = "error"
        elif issues:
            status = "warn"
        else:
            status = "ok"
        return status, warn_codes, issues

    day_start = pd.Timestamp(day_str, tz="UTC")
    day_end = day_start + pd.Timedelta(days=1)

    min_ts = df["timestamp_utc"].min()
    max_ts = df["timestamp_utc"].max()
    if min_ts < day_start or max_ts >= day_end:
        add("ts_out_of_day", f"min={min_ts}, max={max_ts}, expected [{day_start}, {day_end})", "error")

    outside = ((df["timestamp_utc"] < day_start) | (df["timestamp_utc"] >= day_end)).sum()
    if outside > 0:
        add("timestamps_outside_day", f"{outside} rows outside [{day_start}, {day_end})", "error")

    if not df["timestamp_utc"].is_monotonic_increasing:
        add("unsorted_timestamps", "timestamp_utc is not non-decreasing", "error")

    dup_ts = df["timestamp_utc"].duplicated().sum()
    if dup_ts > 100:
        add("high_dup_ts_count", f"dup_ts_count={dup_ts}", "warn")

    for col in ["bid", "ask"]:
        bad = (~df[col].apply(lambda x: pd.notna(x) and x not in (float("inf"), float("-inf")))).sum()
        if bad > 0:
            add("bad_values", f"{bad} NaN/inf in column '{col}'", "error")

    bid_gt_ask = (df["bid"] > df["ask"]).sum()
    if bid_gt_ask > 0:
        add("bid_gt_ask", f"{bid_gt_ask} rows where bid > ask", "error")

    for col in ("bid_volume", "ask_volume"):
        bad = (~df[col].apply(lambda x: pd.notna(x) and x not in (float("inf"), float("-inf")))).sum()
        if bad > 0:
            add("bad_volume_values", f"{bad} NaN/inf in '{col}'", "error")
        neg = (df[col] < 0).sum()
        if neg > 0:
            add("negative_volume", f"{neg} rows with negative '{col}'", "error")

    spread = df["ask"] - df["bid"]
    spread_warn = (spread > SPREAD_MAX_OK).sum()
    spread_err = (spread > SPREAD_MAX_ERR).sum()
    if spread_err > 0:
        add(
            "spread_too_large",
            f"{spread_err} rows with spread > {SPREAD_MAX_ERR} ({SPREAD_MAX_ERR * 1e4:.0f} pips)",
            "error",
        )
    elif spread_warn > 0:
        add(
            "spread_elevated",
            f"{spread_warn} rows with spread > {SPREAD_MAX_OK} ({SPREAD_MAX_OK * 1e4:.0f} pips)",
            "warn",
        )

    price_bad = ((df["bid"] < PRICE_MIN) | (df["ask"] > PRICE_MAX)).sum()
    if price_bad > 0:
        add("price_out_of_bounds", f"{price_bad} rows outside [{PRICE_MIN}, {PRICE_MAX}]", "error")

    status = "ok"
    if any(i["severity"] == "error" for i in issues):
        status = "error"
    elif any(i["severity"] == "warn" for i in issues):
        status = "warn"

    return status, warn_codes, issues

## 1) Extension: one Parquet for `EXTENSION_START` … `EXTENSION_END`

Downloads each UTC day in range, runs `validate_day`, drops duplicate timestamps per day, concatenates in date order, writes `EXT_PARQUET`.

In [ ]:
ext_dates = [d.strftime("%Y-%m-%d") for d in pd.date_range(EXTENSION_START, EXTENSION_END, freq="D")]
parts: list[pd.DataFrame] = []
ext_report: list[dict] = []

for day_str in tqdm(ext_dates, desc="Extension"):
    df, missing_hours = download_day(SYMBOL, day_str)
    dup = int(df["timestamp_utc"].duplicated().sum())
    if dup:
        df = df.drop_duplicates(subset=["timestamp_utc"], keep="first").reset_index(drop=True)
    status, _warns, issues = validate_day(df, day_str, missing_hours, cal)
    ext_report.extend(issues)
    print(f"[{status:4}] {day_str}  rows={len(df):8,}  missing_h={len(missing_hours):2}  dupes_dropped={dup}")
    if len(df):
        parts.append(df)

if not parts:
    raise RuntimeError("No extension rows; check range / network.")

ext_df = pd.concat(parts, ignore_index=True)
ext_df = ext_df.sort_values("timestamp_utc").reset_index(drop=True)
if not ext_df["timestamp_utc"].is_monotonic_increasing:
    raise RuntimeError("Extension frame not sorted by time")

ext_df.to_parquet(EXT_PARQUET, index=False)
print(f"Wrote {EXT_PARQUET}  rows={len(ext_df):,}")

if ext_report:
    rpt = pd.DataFrame(ext_report)
    rpath = SCRATCH_DIR / "extension_validation_issues.csv"
    rpt.to_csv(rpath, index=False)
    print(f"Validation issues logged: {rpath} ({len(rpt)} rows)")

## 2) Gap fill: 2021-06-10 + downloaded 2021-06-11 + 2021-06-12

Loads neighbor dailies from `OUT_DIR` (same naming as main ingest). Order is **before → gap day (fresh download) → after**, then sort and write `GAP_PARQUET` under `scratch_notebook_outputs/` (still no change to main manifest/report).

If those neighbor files are missing, run `scripts/split_combined_ticks_to_dailies.py` on your combined Parquet first, or edit the paths in the next cell.

In [ ]:
NEIGHBOR_PARQUET_BEFORE = OUT_DIR / f"EURUSD_{NEIGHBOR_BEFORE}_ticks_utc.parquet"
NEIGHBOR_PARQUET_AFTER = OUT_DIR / f"EURUSD_{NEIGHBOR_AFTER}_ticks_utc.parquet"

for p in (NEIGHBOR_PARQUET_BEFORE, NEIGHBOR_PARQUET_AFTER):
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing neighbor daily: {p}\n"
            "Provide files from split_combined_ticks_to_dailies or re-ingest with dailies kept."
        )

df_before = pd.read_parquet(NEIGHBOR_PARQUET_BEFORE)
df_after = pd.read_parquet(NEIGHBOR_PARQUET_AFTER)
df_before["timestamp_utc"] = pd.to_datetime(df_before["timestamp_utc"], utc=True)
df_after["timestamp_utc"] = pd.to_datetime(df_after["timestamp_utc"], utc=True)

df_gap, missing_hours = download_day(SYMBOL, GAP_DAY)
dup = int(df_gap["timestamp_utc"].duplicated().sum())
if dup:
    df_gap = df_gap.drop_duplicates(subset=["timestamp_utc"], keep="first").reset_index(drop=True)
status, _w, issues = validate_day(df_gap, GAP_DAY, missing_hours, cal)
print(f"[{status}] {GAP_DAY} rows={len(df_gap):,} missing_h={len(missing_hours)} dupes_dropped={dup}")
if issues:
    pd.DataFrame(issues).to_csv(SCRATCH_DIR / "gap_day_validation_issues.csv", index=False)

merged = pd.concat([df_before, df_gap, df_after], ignore_index=True)
merged = merged.sort_values("timestamp_utc").reset_index(drop=True)

# Safety: no duplicate timestamps across the seam (re-downloaded gap vs stale overlap)
ndup = int(merged["timestamp_utc"].duplicated().sum())
if ndup:
    print(f"[WARN] dropping {ndup} duplicate timestamp rows after merge")
    merged = merged.drop_duplicates(subset=["timestamp_utc"], keep="first").reset_index(drop=True)

if not merged["timestamp_utc"].is_monotonic_increasing:
    raise RuntimeError("Merged gap-fill frame not sorted")

merged.to_parquet(GAP_PARQUET, index=False)
print(f"Wrote {GAP_PARQUET}  rows={len(merged):,}")